# Inspect Yahoo Raw Market Data

这个 notebook 直接从 Yahoo Finance 下载原始市场数据，只做展示和检查，不会覆盖本地任何 CSV 文件。

In [ ]:
from io import StringIO
from pathlib import Path

import pandas as pd
import yfinance as yf
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)


In [ ]:
TICKERS = ["SPY", "TLT", "GLD", "UUP", "HYG", "LQD", "^VIX"]
START_DATE = "2015-01-01"
AUTO_ADJUST = True

display(Markdown(f"**Tickers:** `{TICKERS}`"))
display(Markdown(f"**Start date:** `{START_DATE}`"))
display(Markdown("**Note:** this notebook only downloads and displays data. It does **not** overwrite local files."))


## Ticker Guide

下面这些 ticker 是这个项目里常用的跨资产代理:

- `SPY`: S&P 500 ETF, 代表美国股票市场
- `TLT`: 长久期美国国债 ETF, 常用来代表避险利率资产
- `GLD`: 黄金 ETF, 常用来代表贵金属避险资产
- `UUP`: 美元指数代理 ETF, 用来近似美元强弱
- `HYG`: 高收益信用债 ETF, 常用来代表风险信用资产
- `LQD`: 投资级信用债 ETF, 常用来和 `HYG` 一起做信用利差代理
- `^VIX`: CBOE VIX 指数, 代表隐含波动率和市场恐慌程度

后续如果 `LQD` 为空, 会直接影响:

- `credit_spread_proxy = log(HYG / LQD)`
- `LQD_ret`
- `corr(HYG_ret, LQD_ret)`

In [ ]:
raw = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    auto_adjust=AUTO_ADJUST,
    progress=False,
)

display(Markdown("## Raw Download Result"))
display(raw.head())


In [ ]:
display(Markdown("## Raw Metadata"))

if isinstance(raw.columns, pd.MultiIndex):
    top_level_fields = list(raw.columns.get_level_values(0).unique())
    ticker_level = list(raw.columns.get_level_values(1).unique())
else:
    top_level_fields = list(raw.columns)
    ticker_level = []

metadata_df = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "column_count",
            "index_start",
            "index_end",
            "is_multiindex_columns",
        ],
        "value": [
            len(raw),
            raw.shape[1],
            raw.index.min(),
            raw.index.max(),
            isinstance(raw.columns, pd.MultiIndex),
        ],
    }
)

display(metadata_df)
display(Markdown(f"**Top-level fields:** `{top_level_fields}`"))
if ticker_level:
    display(Markdown(f"**Ticker level:** `{ticker_level}`"))


In [ ]:
display(Markdown("## Field and Metric Explanation"))

field_guide = pd.DataFrame(
    {
        "item": [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",
            "row_count",
            "column_count",
            "index_start / index_end",
            "is_multiindex_columns",
            "non_null_count",
            "missing_count",
            "missing_pct",
        ],
        "meaning": [
            "当日开盘价。开启 auto_adjust=True 时, 价格通常已经过分红拆股调整。",
            "当日最高价。",
            "当日最低价。",
            "当日收盘价。项目里最常用的是这个面板。",
            "成交量。指数类 ticker 可能没有常规成交量含义。",
            "总共有多少个交易日样本。",
            "下载结果总共有多少列。多 ticker 下载时通常是 字段数 x ticker数。",
            "时间索引的起止日期。",
            "列是否是 MultiIndex。Yahoo 多 ticker 下载时通常会是两层列索引。",
            "某一列有多少个非空值。",
            "某一列有多少个空值。",
            "某一列空值占全部样本的比例。",
        ],
    }
)

display(field_guide)

display(Markdown("如果你后面是做 Track A / Track B, 一般最关心的是 `Close` 面板里的价格列, 因为收益率和 rolling feature 都是从这里构造出来的。"))


In [ ]:
display(Markdown("## Close Panel"))

if isinstance(raw.columns, pd.MultiIndex):
    if "Close" in raw.columns.get_level_values(0):
        close = raw["Close"].copy()
    else:
        close = raw.xs("Close", axis=1, level=0).copy()
else:
    close = raw.copy()

close.index.name = "Date"
display(close.head())

buffer = StringIO()
close.info(buf=buffer)
print(buffer.getvalue())


In [ ]:
display(Markdown("## Missing Values by Column"))

missing_summary = pd.DataFrame(
    {
        "column": close.columns,
        "non_null_count": close.notna().sum().values,
        "missing_count": close.isna().sum().values,
        "missing_pct": (close.isna().mean() * 100).round(2).values,
    }
).sort_values(["missing_count", "column"], ascending=[False, True]).reset_index(drop=True)

display(missing_summary)


In [ ]:
display(Markdown("## LQD Check"))

if "LQD" not in close.columns:
    raise KeyError("`LQD` not found in the downloaded Close panel.")

lqd_summary = pd.DataFrame(
    {
        "metric": [
            "lqd_non_null_count",
            "lqd_missing_count",
            "lqd_is_all_null",
            "lqd_first_valid_date",
            "lqd_last_valid_date",
        ],
        "value": [
            int(close["LQD"].notna().sum()),
            int(close["LQD"].isna().sum()),
            bool(close["LQD"].isna().all()),
            close["LQD"].first_valid_index(),
            close["LQD"].last_valid_index(),
        ],
    }
)

display(lqd_summary)
display(close[["HYG", "LQD"]].head(15))
display(close[["HYG", "LQD"]].tail(15))

if close["LQD"].isna().all():
    display(Markdown("### Result: the raw Yahoo download also returns `LQD` as all null in this run."))
else:
    display(Markdown("### Result: the raw Yahoo download returns non-null `LQD` values in this run."))


In [ ]:
display(Markdown("## Compare With Existing Local market_data.csv (read-only)"))

local_candidates = [
    Path.cwd() / "data" / "market_data.csv",
    Path.cwd() / "Final_Project" / "data" / "market_data.csv",
]
local_market_path = next((path for path in local_candidates if path.exists()), None)

if local_market_path is None:
    display(Markdown("No local `market_data.csv` found from the current working directory."))
else:
    local_market = pd.read_csv(local_market_path, index_col="Date", parse_dates=True)
    compare_df = pd.DataFrame(
        {
            "source": ["local_market_data.csv", "raw_yahoo_close"],
            "lqd_non_null_count": [
                int(local_market["LQD"].notna().sum()) if "LQD" in local_market.columns else None,
                int(close["LQD"].notna().sum()),
            ],
            "lqd_missing_count": [
                int(local_market["LQD"].isna().sum()) if "LQD" in local_market.columns else None,
                int(close["LQD"].isna().sum()),
            ],
            "start_date": [local_market.index.min(), close.index.min()],
            "end_date": [local_market.index.max(), close.index.max()],
        }
    )
    display(Markdown(f"**Local file:** `{local_market_path}`"))
    display(compare_df)
